In [1]:
import pandas as pd

# 读取 HGNC 下载文件（tab 分隔）
df = pd.read_csv("custom.txt", sep="\t")

print("原始数据维度:", df.shape)
print("列名:")
print(df.columns.tolist())

# 先尽量自动识别“Gene group ID / Gene group name”列
col_id = None
col_name = None

for c in df.columns:
    c_low = c.strip().lower()
    if "gene group id" in c_low or c_low == "family.id":
        col_id = c
    if "gene group name" in c_low or c_low == "family.name":
        col_name = c

print("识别到的 Gene group ID 列:", col_id)
print("识别到的 Gene group name 列:", col_name)

# 优先按 group id = 471 筛选；如果没有就按 group name = CD molecules 筛选
if col_id is not None:
    df_cd = df[df[col_id].astype(str).str.contains(r"\b471\b", na=False)].copy()
elif col_name is not None:
    df_cd = df[df[col_name].astype(str).str.contains(r"^CD molecules$", case=False, na=False)].copy()
else:
    raise ValueError(
        "没有找到 Gene group ID 或 Gene group name 列。请先 print(df.columns.tolist()) 检查列名。"
    )

print("筛选后维度:", df_cd.shape)

# 可选：把列名改得更接近 notebook 里的名字
rename_map = {}
for c in df_cd.columns:
    c_low = c.strip().lower()
    if c_low == "hgnc id":
        rename_map[c] = "HGNC ID (gene)"
    elif c_low == "approved symbol":
        rename_map[c] = "Approved symbol"
    elif c_low == "approved name":
        rename_map[c] = "Approved name"
    elif c_low == "previous symbols":
        rename_map[c] = "Previous symbols"
    elif c_low == "alias symbols":
        rename_map[c] = "Aliases"
    elif c_low == "chromosome":
        rename_map[c] = "Chromosome"

df_cd = df_cd.rename(columns=rename_map)

# 保存
df_cd.to_csv("group-471.csv", index=False)

print("已保存为 group-471.csv")
print(df_cd.head())

原始数据维度: (50268, 11)
列名:
['HGNC ID', 'Approved symbol', 'Approved name', 'Status', 'Previous symbols', 'Alias symbols', 'Chromosome', 'Accession numbers', 'RefSeq IDs', 'Gene group ID', 'Gene group name']
识别到的 Gene group ID 列: Gene group ID
识别到的 Gene group name 列: Gene group name
筛选后维度: (394, 11)
已保存为 group-471.csv
    HGNC ID (gene) Approved symbol  \
79         HGNC:40           ABCB1   
131        HGNC:74           ABCG2   
236      HGNC:2707             ACE   
254      HGNC:4035           ACKR1   
463       HGNC:215           ADAM8   

                                         Approved name    Status  \
79           ATP binding cassette subfamily B member 1  Approved   
131  ATP binding cassette subfamily G member 2 (JR ...  Approved   
236                    angiotensin I converting enzyme  Approved   
254  atypical chemokine receptor 1 (Duffy blood group)  Approved   
463                     ADAM metallopeptidase domain 8  Approved   

     Previous symbols                         